# Notebook 06 — SVD and Its Connection to PCA

**Module:** 04 — Linear Algebra  
**Notebook:** 06 of 10  
**Prerequisites:** NB04, NB05  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

The covariance matrix approach to PCA (NB05) fails in practice when genes >> samples:
computing a 20,000 × 20,000 covariance matrix is expensive. SVD solves this directly
on the data matrix — it's more numerically stable and computationally efficient.
It also generalizes beyond PCA: low-rank approximation, image compression, and
collaborative filtering all use SVD.

---
## Step 2 — Intuition

Every matrix $\mathbf{X}$ can be factored as:
$$\mathbf{X} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^\top$$

- $\mathbf{U}$ (left singular vectors): patterns across samples — the PC scores
- $\boldsymbol{\Sigma}$ (singular values on diagonal): how important each pattern is
- $\mathbf{V}$ (right singular vectors): patterns across genes — the PC loadings

Think of it as: $\mathbf{X}$ = (sample patterns) × (importance weights) × (gene patterns).

**Connection to PCA:** the right singular vectors $\mathbf{V}$ are the eigenvectors
of $\mathbf{X}^\top \mathbf{X}$ — exactly the PCA loadings.

---
## Step 3 — Biological Background

**SVD in genomics (Alter et al., 2000):**
- $\mathbf{U}$ columns = 'eigenarrays': characteristic expression patterns across samples
- $\mathbf{V}$ columns = 'eigengenes': characteristic patterns across genes
- Truncated SVD (keep top $k$) = low-rank approximation, filtering biological noise

**Single-cell RNA-seq:**
- Scanpy uses truncated SVD (via `scipy.sparse.linalg.svds`) on sparse count matrices
  — it's faster than covariance-based PCA for millions of cells
- UMAP and Leiden clustering both operate on the top 50 SVD components

**Low-rank approximation:**
Keep only the top $k$ singular values/vectors:
$$\mathbf{X} \approx \mathbf{U}_k \boldsymbol{\Sigma}_k \mathbf{V}_k^\top$$
This is the best rank-$k$ approximation in Frobenius norm (Eckart-Young theorem).

---
## Step 4 — Mathematical Explanation

**Full SVD:** $\mathbf{X} \in \mathbb{R}^{m \times n}$:
$$\mathbf{X} = \mathbf{U}_{m\times m} \boldsymbol{\Sigma}_{m\times n} \mathbf{V}^\top_{n\times n}$$
- $\mathbf{U}$: orthonormal, columns are left singular vectors
- $\boldsymbol{\Sigma}$: diagonal, non-negative singular values $\sigma_1 \geq \sigma_2 \geq \ldots \geq 0$
- $\mathbf{V}$: orthonormal, columns are right singular vectors

**Connection to eigendecomposition:**
- $\mathbf{X}^\top \mathbf{X} = \mathbf{V}\boldsymbol{\Sigma}^2\mathbf{V}^\top$ → $\mathbf{V}$ = eigenvectors of $\mathbf{X}^\top \mathbf{X}$
- $\mathbf{X}\mathbf{X}^\top = \mathbf{U}\boldsymbol{\Sigma}^2\mathbf{U}^\top$ → $\mathbf{U}$ = eigenvectors of $\mathbf{X}\mathbf{X}^\top$
- Eigenvalues of $\mathbf{X}^\top\mathbf{X}$ are $\sigma_i^2$ (singular values squared)

**Eckart-Young theorem:**
$$\min_{\text{rank}(\hat{\mathbf{X}})=k} \|\mathbf{X} - \hat{\mathbf{X}}\|_F = \sqrt{\sum_{i>k}\sigma_i^2}$$
The minimum is achieved by the truncated SVD $\hat{\mathbf{X}} = \mathbf{U}_k \boldsymbol{\Sigma}_k \mathbf{V}_k^\top$.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA as sklearn_PCA

In [ ]:
# Cell 6.1 — SVD from scratch using np.linalg.svd + verify PCA equivalence
rng = np.random.default_rng(42)
n_samples, n_genes = 90, 500
cell_type = np.repeat([0, 1, 2], 30)
type_profiles = rng.normal(0, 3, (3, n_genes))
X = type_profiles[cell_type] + rng.normal(0, 0.8, (n_samples, n_genes))
X_centered = X - X.mean(axis=0)

# SVD of the centered data matrix
U, s, Vt = np.linalg.svd(X_centered, full_matrices=False)
# U: (samples, min(m,n)), s: (min,), Vt: (min, genes)

print(f"U shape: {U.shape}  (left singular vectors — sample patterns)")
print(f"s shape: {s.shape}  (singular values)")
print(f"Vt shape: {Vt.shape} (right singular vectors — gene patterns)")

# SVD reconstruction: verify X_centered ≈ U @ diag(s) @ Vt
X_reconstructed = U @ np.diag(s) @ Vt
print(f"\nReconstruction error: {np.linalg.norm(X_centered - X_reconstructed):.2e}  (should be ~0)")

# PC scores via SVD = U @ diag(s) (first k columns)
k = 5
scores_svd = U[:, :k] * s[:k]   # equivalent to X_centered @ Vt[:k].T

# Compare to sklearn PCA scores
sk = sklearn_PCA(n_components=k, random_state=42).fit(X)
sk_scores = sk.transform(X)

print(f"\nSVD vs sklearn PC score correlation:")
for i in range(k):
    r = np.abs(np.corrcoef(scores_svd[:, i], sk_scores[:, i])[0, 1])
    print(f"  PC{i+1}: |r| = {r:.8f}")

In [ ]:
# Cell 6.2 — Low-rank approximation: information compression
def low_rank_approx(X: np.ndarray, k: int) -> np.ndarray:
    """Best rank-k approximation of X via truncated SVD."""
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

# Compression quality vs. rank k
frobenius_X = np.linalg.norm(X_centered, 'fro')
print(f"{'Rank k':>8} {'Approx error':>15} {'% variance explained':>22}")
print("-" * 50)
for k in [1, 2, 3, 5, 10, 20, 50, 90]:
    X_approx = low_rank_approx(X_centered, k)
    error = np.linalg.norm(X_centered - X_approx, 'fro')
    variance_explained = 1 - (error / frobenius_X)**2
    print(f"  k={k:>5}    {error:>13.3f}   {variance_explained:>20.2%}")

In [ ]:
# Cell 6.3 — Singular value spectrum and explained variance
U_full, s_full, Vt_full = np.linalg.svd(X_centered, full_matrices=False)
# Singular values relate to eigenvalues: lambda_i = sigma_i^2 / (n-1)
eigenvalues_from_svd = s_full**2 / (n_samples - 1)
explained_var_svd = s_full**2 / (s_full**2).sum()
print("Singular values squared ↔ eigenvalues of covariance matrix:")
print(f"  From SVD: {eigenvalues_from_svd[:5].round(4)}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Singular value spectrum + low-rank approximation quality
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: Singular value spectrum
ax = axes[0]
ax.plot(range(1, len(s_full)+1), s_full, 'o-', color='steelblue', ms=4, lw=1.5)
ax.set_xlabel('Singular value index')
ax.set_ylabel('Singular value σ')
ax.set_title('Singular value spectrum\n(elbow at index 3 = 3 cell types)')
ax.set_xlim(0.5, 30)

# Panel 2: Cumulative variance explained via SVD
ax = axes[1]
cum_var = np.cumsum(explained_var_svd)
ax.plot(range(1, len(cum_var)+1), cum_var, color='steelblue', lw=2)
ax.axhline(0.9, color='gray', ls='--', lw=1)
ax.set_xlim(0, 30)
ax.set_xlabel('Number of singular values k')
ax.set_ylabel('Cumulative variance explained')
ax.set_title('SVD: cumulative explained variance')

# Panel 3: Low-rank reconstruction of expression matrix (visual)
ax = axes[2]
k_vals = [1, 3, 10, 30]
errors = [np.linalg.norm(X_centered - low_rank_approx(X_centered, k), 'fro')
          for k in k_vals]
relative_errors = [e / frobenius_X for e in errors]
ax.bar([str(k) for k in k_vals], [1 - re**2 for re in relative_errors],
       color='steelblue', alpha=0.8)
ax.set_xlabel('Rank k'); ax.set_ylabel('Variance captured (Frobenius)')
ax.set_title('Low-rank approximation quality')

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Verify the Eckart-Young theorem numerically: compute `‖X - X_rank_k‖_F²` and
   verify it equals `∑_{i>k} σᵢ²`. Show the algebra connecting them.
2. The right singular vectors Vt are the PCA loadings. Verify that `Vt[:2] @ Vt[:2].T`
   is the identity (orthonormality of singular vectors).
3. Why is SVD-based PCA preferred over covariance-based PCA when genes >> samples?
   Compute the computational cost of each approach in big-O notation.
4. Apply low-rank approximation to the expression matrix and visualize the original
   vs. rank-3 approximation as a heatmap. What structure is preserved? What is lost?

---
## Quiz — Active Recall

1. Write the SVD decomposition $\mathbf{X} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^\top$.
   What are the shapes of each matrix?
2. How are singular values related to eigenvalues of $\mathbf{X}^\top\mathbf{X}$?
3. What is the Eckart-Young theorem? State it in one sentence.
4. How do you compute PC scores from the SVD output? Write the formula.
5. What does truncated SVD (keep top $k$) correspond to in biological terms?

---
## Papers Referenced

- Alter, Brown & Botstein (2000). DOI: 10.1073/pnas.97.18.10101

---
## Reflection

**Date completed:** ____________________

> *[Can you explain the SVD decomposition X = UΣVᵀ to a biologist, specifying what U, Σ, and V each represent?]*

---
*Next: `07_matrices_as_graphs_adjacency_laplacian.ipynb`*